# Business Enrichment from `business_input.csv`

This is the main workflow notebook.

It reads a list of businesses from `business_input.csv` and generates output columns step-by-step:
- Company Description
- Company Address
- Company Telephone
- Company Email
- Company House Number
- Company House URL

Input CSV required columns:
- `name`
- `location`
- `website`

In [2]:
import json
import re
import time
from typing import List
from urllib.parse import quote_plus, urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

cell_start = time.perf_counter()
print("[Cell 1] Starting input load and dataframe initialization...")

OUTPUT_COLUMNS = [
    "Company Name",
    "Company Description",
    "Company Area",
    "Company URL",
    "Company Address",
    "Company Telephone",
    "Company Email",
    "Company House Number",
    "Company House URL",
]

input_path = "business_input.csv"
print(f"[Cell 1] Reading input CSV: {input_path}")
input_df = pd.read_csv(input_path)
required = {"company name", "company url"}
missing = required.difference({c.lower() for c in input_df.columns})
if missing:
    raise RuntimeError(
        "Missing required input columns. Expected: ['Company Name', 'Company URL']"
    )

col_map = {c.lower(): c for c in input_df.columns}
result_df = pd.DataFrame(
    {
        "Company Name": input_df[col_map["company name"]].astype(str).str.strip(),
        "Company Description": "",
        "Company Area": "",
        "Company URL": input_df[col_map["company url"]].astype(str).str.strip(),
        "Company Address": "",
        "Company Telephone": "",
        "Company Email": "",
        "Company House Number": "",
        "Company House URL": "",
    }
)

elapsed = time.perf_counter() - cell_start
print(f"[Cell 1] Loaded {len(result_df)} businesses in {elapsed:.2f}s")
result_df[["Company Name", "Company URL"]].head(10)

[Cell 1] Starting input load and dataframe initialization...
[Cell 1] Reading input CSV: business_input.csv


RuntimeError: Missing required input columns. Expected: ['Company Name', 'Company URL']

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; BusinessNotebook/1.0)"}
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

# In-memory caches to avoid repeated network calls per URL/name.
html_cache = {}
context_cache = {}
about_cache = {}
company_house_cache = {}


def fetch_html(url: str, timeout: int = 12) -> str:
    if url in html_cache:
        return html_cache[url]

    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        if response.ok and response.text:
            html_cache[url] = response.text
            return response.text
    except requests.RequestException:
        pass

    html_cache[url] = ""
    return ""


def html_to_text(html: str, max_chars: int = 8000) -> str:
    if not html:
        return ""
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars]


def discover_contact_urls(base_url: str, homepage_html: str = "") -> List[str]:
    urls = [
        urljoin(base_url.rstrip("/") + "/", "contact"),
        urljoin(base_url.rstrip("/") + "/", "contact-us"),
        urljoin(base_url.rstrip("/") + "/", "contacts"),
        urljoin(base_url.rstrip("/") + "/", "get-in-touch"),
    ]
    homepage = homepage_html or fetch_html(base_url)
    if homepage:
        soup = BeautifulSoup(homepage, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a.get("href", "").strip()
            label = (a.get_text(" ", strip=True) or "").lower()
            if "contact" in href.lower() or "contact" in label:
                urls.append(urljoin(base_url, href))

    unique = []
    seen = set()
    for u in urls:
        if u not in seen:
            unique.append(u)
            seen.add(u)

    # Limit extra pages to reduce latency.
    return unique[:4]


def discover_about_url(base_url: str, homepage_html: str = "") -> str:
    candidates = [
        urljoin(base_url.rstrip("/") + "/", "about"),
        urljoin(base_url.rstrip("/") + "/", "about-us"),
        urljoin(base_url.rstrip("/") + "/", "our-story"),
        urljoin(base_url.rstrip("/") + "/", "who-we-are"),
    ]

    homepage = homepage_html or fetch_html(base_url)
    if homepage:
        soup = BeautifulSoup(homepage, "html.parser")
        for a in soup.find_all("a", href=True):
            href = a.get("href", "").strip()
            label = (a.get_text(" ", strip=True) or "").lower()
            if "about" in href.lower() or "about" in label:
                candidates.insert(0, urljoin(base_url, href))

    seen = set()
    for url in candidates:
        if url in seen:
            continue
        seen.add(url)
        html = fetch_html(url)
        text = html_to_text(html, max_chars=6000)
        if text:
            return url
    return ""


def collect_about_text(website_url: str) -> str:
    if website_url in about_cache:
        return about_cache[website_url]

    homepage_html = fetch_html(website_url)
    about_url = discover_about_url(website_url, homepage_html=homepage_html)
    if not about_url:
        about_cache[website_url] = ""
        return ""

    about_html = fetch_html(about_url)
    about_text = html_to_text(about_html, max_chars=6000)
    about_cache[website_url] = about_text or ""
    return about_cache[website_url]


def summarize_about_text(about_text: str, max_sentences: int = 2) -> str:
    text = re.sub(r"\s+", " ", (about_text or "").strip())
    if not text:
        return ""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    if not sentences:
        return ""
    summary = " ".join(sentences[:max_sentences]).strip()
    if summary and summary[-1] not in ".!?":
        summary += "."
    return summary


def call_ollama(prompt: str, timeout: int = 90) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": 220},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json().get("response", "").strip()


def call_ollama_json(prompt: str, default: dict, timeout: int = 90) -> dict:
    try:
        raw = call_ollama(prompt, timeout=timeout)
        candidate = raw
        if "{" in raw and "}" in raw:
            candidate = raw[raw.find("{") : raw.rfind("}") + 1]
        data = json.loads(candidate)
        if isinstance(data, dict):
            return data
    except Exception:
        pass
    return default


def extract_field_with_llama(company_name: str, field_name: str, website_text: str) -> str:
    prompt = f"""
You extract business profile fields from website text.
Return ONLY valid JSON: {{"value":"..."}}.
If unknown, return {{"value":""}}.

Field requested: {field_name}
Company name: {company_name}

Rules:
- company_area: return exactly "Real Estate" if business is estate agency/property sales/lettings; otherwise return ""
- company_address: full postal address only
- company_telephone: UK phone format if present
- company_email: business contact email only
- Do not invent facts. Use only provided text.

Website text:
{website_text}
""".strip()

    result = call_ollama_json(prompt, default={"value": ""})
    value = str(result.get("value", "") or "").strip()
    return value


def extract_company_house_with_llama(company_name: str, company_house_text: str) -> dict:
    prompt = f"""
You extract Companies House mapping for one business from search/page text.
Return ONLY valid JSON with keys exactly:
{{"company_house_number":"","company_house_url":""}}
If uncertain, leave both empty.

Constraints:
- Name must be identical or very high similarity to: {company_name}
- Nature of business must indicate real estate agency activity (e.g., SIC 68310 or clearly related)
- Do not invent values; only use provided text.

Companies House text:
{company_house_text}
""".strip()

    result = call_ollama_json(
        prompt,
        default={"company_house_number": "", "company_house_url": ""},
        timeout=120,
    )
    return {
        "company_house_number": str(result.get("company_house_number", "") or "").strip(),
        "company_house_url": str(result.get("company_house_url", "") or "").strip(),
    }


def collect_context_text(website_url: str) -> str:
    if website_url in context_cache:
        return context_cache[website_url]

    parts = []
    home_html = fetch_html(website_url)
    home_text = html_to_text(home_html)
    if home_text:
        parts.append("[HOMEPAGE]\n" + home_text)

    for u in discover_contact_urls(website_url, homepage_html=home_html):
        page_html = fetch_html(u)
        page_text = html_to_text(page_html, max_chars=5000)
        if page_text:
            parts.append(f"[CONTACT_PAGE] {u}\n" + page_text)

    context_cache[website_url] = "\n\n".join(parts)[:12000]
    return context_cache[website_url]


def collect_company_house_text(company_name: str) -> str:
    if company_name in company_house_cache:
        return company_house_cache[company_name]

    search_url = (
        "https://find-and-update.company-information.service.gov.uk/search/companies?q="
        + quote_plus(company_name)
    )
    search_html = fetch_html(search_url)
    search_text = html_to_text(search_html, max_chars=12000)
    if not search_text:
        company_house_cache[company_name] = ""
        return ""

    company_house_cache[company_name] = f"[SEARCH_URL] {search_url}\n{search_text}"
    return company_house_cache[company_name]

In [ ]:
# Ollama health check (safe to run independently)
import requests
import time

cell_start = time.perf_counter()
print("[Cell 3] Starting Ollama health check...")

check_model = globals().get("OLLAMA_MODEL", "llama3.2:3b")
candidate_bases = ["http://localhost:11434", "http://127.0.0.1:11434"]
last_error = None

for base_url in candidate_bases:
    base_start = time.perf_counter()
    print(f"[Cell 3] Trying endpoint: {base_url}")
    try:
        tags_resp = requests.get(f"{base_url}/api/tags", timeout=20)
        tags_resp.raise_for_status()
        tags_data = tags_resp.json()
        installed = [m.get("name", "") for m in tags_data.get("models", [])]

        if not any(name == check_model or name.startswith(check_model + ":") for name in installed):
            raise RuntimeError(
                f"Ollama is running, but model '{check_model}' is not installed. "
                f"Installed models: {installed or 'none'}. Run: ollama pull {check_model}"
            )

        health_payload = {
            "model": check_model,
            "prompt": "Reply with exactly: OK",
            "stream": False,
            "options": {"temperature": 0},
        }

        health_response = requests.post(f"{base_url}/api/generate", json=health_payload, timeout=120)
        health_response.raise_for_status()
        health_text = health_response.json().get("response", "").strip()

        OLLAMA_URL = f"{base_url}/api/generate"

        base_elapsed = time.perf_counter() - base_start
        total_elapsed = time.perf_counter() - cell_start
        print(f"[Cell 3] Ollama reachable at {base_url}")
        print(f"[Cell 3] Model available: {check_model}")
        print(f"[Cell 3] Test response: {health_text}")
        print(f"[Cell 3] Endpoint check time: {base_elapsed:.2f}s")
        print(f"[Cell 3] Total cell time: {total_elapsed:.2f}s")
        break
    except Exception as exc:
        last_error = exc
        print(f"[Cell 3] Failed on {base_url}: {exc}")
else:
    raise RuntimeError(
        "Ollama health check failed for localhost and 127.0.0.1. "
        f"Last error: {last_error}"
    )

In [ ]:
# Generate Company Description (About page only, non-LLM 2-sentence summary) with progress/timing logs
import time

total_start = time.perf_counter()
row_count = len(result_df)
print(f"Starting description generation for {row_count} companies...")

for i, row in result_df.iterrows():
    name = row["Company Name"]
    url = row["Company URL"]
    row_start = time.perf_counter()

    print(f"[{i + 1}/{row_count}] {name} -> collecting ABOUT page from {url}")
    fetch_start = time.perf_counter()
    about_text = collect_about_text(url)
    fetch_seconds = time.perf_counter() - fetch_start

    if not about_text:
        print(f"    no about page text found ({fetch_seconds:.2f}s), skipping")
        continue

    summary_start = time.perf_counter()
    value = summarize_about_text(about_text, max_sentences=2)
    summary_seconds = time.perf_counter() - summary_start

    result_df.at[i, "Company Description"] = value
    total_row_seconds = time.perf_counter() - row_start
    print(
        f"    done in {total_row_seconds:.2f}s "
        f"(about fetch {fetch_seconds:.2f}s, summarize {summary_seconds:.2f}s)"
    )

elapsed = time.perf_counter() - total_start
print(f"Completed description generation in {elapsed:.2f}s")

result_df[["Company Name", "Company Description"]]

In [ ]:
# Generate Company Area (via Llama classification) with progress/timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 6] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
row_count = len(result_df)
print(f"[Cell 6] Starting Company Area generation for {row_count} companies...")

for i, row in result_df.iterrows():
    row_start = time.perf_counter()
    name = row["Company Name"]
    cached = row["Company URL"] in context_cache
    print(f"[Cell 6][{i + 1}/{row_count}] {name} -> collecting context (cache={'hit' if cached else 'miss'})")

    context_start = time.perf_counter()
    context_text = collect_context_text(row["Company URL"])
    context_elapsed = time.perf_counter() - context_start
    if not context_text:
        print(f"    no context found ({context_elapsed:.2f}s), skipping")
        continue

    print("    ----- CONTEXT START -----")
    print(context_text)
    print("    ----- CONTEXT END -----")

    llama_start = time.perf_counter()
    value = extract_field_with_llama(name, "company_area", context_text)
    llama_elapsed = time.perf_counter() - llama_start

    result_df.at[i, "Company Area"] = "Real Estate" if value.strip().lower() == "real estate" else ""
    row_elapsed = time.perf_counter() - row_start
    print(f"    done in {row_elapsed:.2f}s (context {context_elapsed:.2f}s, llama {llama_elapsed:.2f}s)")

cell_elapsed = time.perf_counter() - cell_start
print(f"[Cell 6] Completed in {cell_elapsed:.2f}s")

result_df[["Company Name", "Company Area"]]

In [ ]:
# Generate Company Address (via Llama) with progress/timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 7] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
row_count = len(result_df)
print(f"[Cell 7] Starting Company Address generation for {row_count} companies...")

for i, row in result_df.iterrows():
    row_start = time.perf_counter()
    name = row["Company Name"]
    print(f"[Cell 7][{i + 1}/{row_count}] {name} -> collecting context")

    context_start = time.perf_counter()
    context_text = collect_context_text(row["Company URL"])
    context_elapsed = time.perf_counter() - context_start
    if not context_text:
        print(f"    no context found ({context_elapsed:.2f}s), skipping")
        continue

    print("    ----- CONTEXT START -----")
    print(context_text)
    print("    ----- CONTEXT END -----")

    llama_start = time.perf_counter()
    value = extract_field_with_llama(name, "company_address", context_text)
    llama_elapsed = time.perf_counter() - llama_start

    result_df.at[i, "Company Address"] = value
    row_elapsed = time.perf_counter() - row_start
    print(f"    done in {row_elapsed:.2f}s (context {context_elapsed:.2f}s, llama {llama_elapsed:.2f}s)")

cell_elapsed = time.perf_counter() - cell_start
print(f"[Cell 7] Completed in {cell_elapsed:.2f}s")

result_df[["Company Name", "Company Address"]]

In [ ]:
# Generate Company Telephone (via Llama) with progress/timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 8] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
row_count = len(result_df)
print(f"[Cell 8] Starting Company Telephone generation for {row_count} companies...")

for i, row in result_df.iterrows():
    row_start = time.perf_counter()
    name = row["Company Name"]
    cached = row["Company URL"] in context_cache
    print(f"[Cell 8][{i + 1}/{row_count}] {name} -> collecting context (cache={'hit' if cached else 'miss'})")

    context_start = time.perf_counter()
    context_text = collect_context_text(row["Company URL"])
    context_elapsed = time.perf_counter() - context_start
    if not context_text:
        print(f"    no context found ({context_elapsed:.2f}s), skipping")
        continue

    print("    ----- CONTEXT START -----")
    print(context_text)
    print("    ----- CONTEXT END -----")

    llama_start = time.perf_counter()
    value = extract_field_with_llama(name, "company_telephone", context_text)
    llama_elapsed = time.perf_counter() - llama_start

    result_df.at[i, "Company Telephone"] = value
    row_elapsed = time.perf_counter() - row_start
    print(f"    done in {row_elapsed:.2f}s (context {context_elapsed:.2f}s, llama {llama_elapsed:.2f}s)")

cell_elapsed = time.perf_counter() - cell_start
print(f"[Cell 8] Completed in {cell_elapsed:.2f}s")

result_df[["Company Name", "Company Telephone"]]

In [ ]:
# Generate Company Email (via Llama) with progress/timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 9] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
row_count = len(result_df)
print(f"[Cell 9] Starting Company Email generation for {row_count} companies...")

for i, row in result_df.iterrows():
    row_start = time.perf_counter()
    name = row["Company Name"]
    print(f"[Cell 9][{i + 1}/{row_count}] {name} -> collecting context")

    context_start = time.perf_counter()
    context_text = collect_context_text(row["Company URL"])
    context_elapsed = time.perf_counter() - context_start
    if not context_text:
        print(f"    no context found ({context_elapsed:.2f}s), skipping")
        continue

    llama_start = time.perf_counter()
    value = extract_field_with_llama(name, "company_email", context_text)
    llama_elapsed = time.perf_counter() - llama_start

    result_df.at[i, "Company Email"] = value
    row_elapsed = time.perf_counter() - row_start
    print(f"    done in {row_elapsed:.2f}s (context {context_elapsed:.2f}s, llama {llama_elapsed:.2f}s)")

cell_elapsed = time.perf_counter() - cell_start
print(f"[Cell 9] Completed in {cell_elapsed:.2f}s")

result_df[["Company Name", "Company Email"]]

In [ ]:
# Generate Company House Number and Company House URL (via Llama) with progress/timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 10] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
row_count = len(result_df)
print(f"[Cell 10] Starting Company House extraction for {row_count} companies...")

for i, row in result_df.iterrows():
    row_start = time.perf_counter()
    company_name = row["Company Name"]
    cached = company_name in company_house_cache
    print(f"[Cell 10][{i + 1}/{row_count}] {company_name} -> collecting Companies House text (cache={'hit' if cached else 'miss'})")

    ch_start = time.perf_counter()
    ch_text = collect_company_house_text(company_name)
    ch_elapsed = time.perf_counter() - ch_start
    if not ch_text:
        print(f"    no Companies House text found ({ch_elapsed:.2f}s), skipping")
        continue

    print("    ----- COMPANIES HOUSE CONTEXT START -----")
    print(ch_text)
    print("    ----- COMPANIES HOUSE CONTEXT END -----")

    llama_start = time.perf_counter()
    mapped = extract_company_house_with_llama(company_name, ch_text)
    llama_elapsed = time.perf_counter() - llama_start

    result_df.at[i, "Company House Number"] = mapped.get("company_house_number", "")
    result_df.at[i, "Company House URL"] = mapped.get("company_house_url", "")

    row_elapsed = time.perf_counter() - row_start
    print(f"    done in {row_elapsed:.2f}s (fetch {ch_elapsed:.2f}s, llama {llama_elapsed:.2f}s)")

cell_elapsed = time.perf_counter() - cell_start
print(f"[Cell 10] Completed in {cell_elapsed:.2f}s")

result_df[["Company Name", "Company House Number", "Company House URL"]]

In [ ]:
# Final output view and save with timing logs
import time

if "result_df" not in globals():
    raise RuntimeError("[Cell 11] result_df is not defined. Run Cell 1 first.")

cell_start = time.perf_counter()
print("[Cell 11] Preparing final dataframe for export...")
final_df = result_df[OUTPUT_COLUMNS].copy()

csv_path = "estate_agencies_tunbridge_wells_brighton.csv"
xlsx_path = "estate_agencies_tunbridge_wells_brighton.xlsx"

print(f"[Cell 11] Writing CSV -> {csv_path}")
csv_start = time.perf_counter()
final_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
csv_elapsed = time.perf_counter() - csv_start

print(f"[Cell 11] Writing XLSX -> {xlsx_path}")
xlsx_start = time.perf_counter()
final_df.to_excel(xlsx_path, index=False)
xlsx_elapsed = time.perf_counter() - xlsx_start

total_elapsed = time.perf_counter() - cell_start
print(f"[Cell 11] Saved {len(final_df)} rows")
print(f"[Cell 11] CSV write time: {csv_elapsed:.2f}s")
print(f"[Cell 11] XLSX write time: {xlsx_elapsed:.2f}s")
print(f"[Cell 11] Total cell time: {total_elapsed:.2f}s")

final_df.head(20)